In [1]:
import pandas as pd
import requests
import time
import re

In [ ]:
df = pd.read_csv(r"C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\SCRAPER PRENSA\datos\raw\mc-onlinenews-mediacloud-20260704202351-content.csv")

df.shape

(25492, 8)

In [16]:
DICCIONARIO = {

    "xenofobia": {

        # Nivel 1: Solo indica que se habla del tema
        "neutro": [
            "inmigración", "migración", "migrante", "migrantes",
            "inmigrante", "inmigrantes", "refugiado", "refugiados",
            "asilo", "frontera", "integración",
            "diversidad cultural", "multiculturalidad",
            "extranjero", "extranjeros"
        ],

        # Nivel 2: Encuadre conflictivo o problematizador
        "marco_conflictivo": [
            "inmigración ilegal",
            "avalancha migratoria",
            "oleada migratoria",
            "presión migratoria",
            "efecto llamada",
            "saturación",
            "colapso de servicios",
            "crisis migratoria",
            "menas",
            "pateras",
            "cayuco",
            "cayucos",
            "repatriación",
            "expulsión masiva",
            "invasión de inmigrantes",
            "flujo migratorio descontrolado",
            "problema migratorio",
            "carga para el sistema",
            "fronteras abiertas",
            "control de fronteras",
            "efecto frontera",
            "inseguridad asociada a la inmigración",
            "delincuencia importada",
            "prioridad nacional",
            "primero los españoles",
            "preferencia nacional",
            "arraigo nacional"
        ],

        # Nivel 3: Hostilidad explícita
        "hostilidad_explicita": [
            "moro",
            "moros",
            "sudaca",
            "sudacas",
            "invasión migratoria",
            "sustitución demográfica",
            "gran sustitución",
            "great replacement",
            "fuera de españa",
            "quédate en tu país",
            "población autóctona en peligro",
            "remigración",
            "islamización",
            "nos invaden",
            "ilegales"
        ],

        # Nivel 4: Violencia o discriminación
        "violencia_discriminacion": [
            "agresión racista",
            "delito de odio racial",
            "discriminación racial",
            "ataque xenófobo",
            "crimen de odio",
            "violencia racista",
            "incidente racista",
            "insulto racista",
            "denuncia por racismo",
            "expulsión discriminatoria"
        ]
    },

    "lgtbifobia": {

        # Nivel 1: Solo indica que se habla del tema
        "neutro": [
            "lgtbi",
            "lgbt",
            "lgtbiq",
            "gay",
            "lesbiana",
            "lesbianas",
            "bisexual",
            "trans",
            "transexual",
            "transexuales",
            "transgénero",
            "identidad de género",
            "orientación sexual",
            "matrimonio homosexual",
            "pareja homosexual",
            "derechos lgtbi"
        ],

        # Nivel 2: Encuadre conflictivo o problematizador
        "marco_conflictivo": [
            "ideología de género",
            "adoctrinamiento",
            "agenda lgtbi",
            "transactivismo",
            "dictadura woke",
            "ingeniería social",
            "sexualización infantil",
            "borrado de las mujeres",
            "familia natural",
            "familia tradicional",
            "terapia de conversión",
            "lobby lgtb",
            "imposición de género",
            "modelo de familia alternativo",
            "adoctrinamiento sexual",
            "propaganda lgtbi",
            "imposición ideológica",
            "activismo trans"
        ],

        # Nivel 3: Hostilidad explícita
        "hostilidad_explicita": [
            "maricón",
            "maricones",
            "bollera",
            "bolleras",
            "travelo",
            "travelos",
            "perversión sexual",
            "anormalidad",
            "enfermedad mental",
            "desviación sexual",
            "depravación",
            "contra natura",
            "aberración"
        ],

        # Nivel 4: Violencia o discriminación
        "violencia_discriminacion": [
            "agresión homófoba",
            "agresión tránsfoba",
            "lgtbifobia",
            "delito de odio",
            "discriminación lgtbi",
            "paliza homófoba",
            "crimen de odio",
            "violencia homófoba",
            "violencia tránsfoba",
            "denuncia por homofobia",
            "transfobia"
        ]
    }
}

In [17]:
def contar_terminos(texto, lista_terminos):

    texto = str(texto).lower()

    total = 0

    for termino in lista_terminos:

        patron = r'\b' + re.escape(termino.lower()) + r'\b'

        total += len(re.findall(patron, texto))

    return total


In [18]:
def analizar_texto(texto, diccionario):

    resultado = {}

    for tema, categorias in diccionario.items():

        for categoria, terminos in categorias.items():

            nombre_columna = f"{tema}_{categoria}"

            resultado[nombre_columna] = contar_terminos(
                texto,
                terminos
            )

    return resultado

In [19]:
def analizar_dataframe(df, columna_texto):

    resultados = df[columna_texto].apply(
        lambda x: analizar_texto(x, DICCIONARIO)
    )

    resultados = pd.DataFrame(resultados.tolist())

    return pd.concat(
        [df.reset_index(drop=True),
         resultados.reset_index(drop=True)],
        axis=1
    )

In [22]:
PESOS = {
    "neutro": 1,
    "marco_conflictivo": 2,
    "hostilidad_explicita": 3,
    "violencia_discriminacion": 4
}

In [45]:
df_prueba_urls = df_xeno.sample(20, random_state=42).copy()

def extraer_cuerpo_trafilatura(url):
    try:
        descarga = trafilatura.fetch_url(url)
        if descarga is None:
            return None

        texto = trafilatura.extract(
            descarga,
            include_comments=False,
            include_tables=False
        )

        return texto

    except Exception:
        return None